In [0]:
from pyspark.sql.functions import col, to_date, from_unixtime

# formating and setting up prices
custom_schema_price = "datetime LONG, price_amount DOUBLE, sequence_position STRING, business_type STRING, mrid STRING, currency STRING, resolution STRING"

prices_df = (spark.read
            .format("parquet")
            .schema(custom_schema_price)
            .load("/Volumes/energy_intelligence/bronze/landing_zone/data/entsoe/Prices/")
            .withColumn("timestamp", from_unixtime(col("datetime")/1e9).cast("timestamp"))
            .withColumn("date_only", to_date(col("timestamp"))))

display(prices_df)

prices_df.write.mode("overwrite").saveAsTable("energy_intelligence.silver.prices_15n60min")

# formating and setting up load
custom_schema_load = "psr_type STRING, datetime LONG, quantity DOUBLE"
load_df = (spark.read
            .format("parquet")
            .schema(custom_schema_load)
            .load("/Volumes/energy_intelligence/bronze/landing_zone/data/entsoe/load_generation/")
            .withColumn("timestamp", from_unixtime(col("datetime")/1e9).cast("timestamp"))
            .withColumn("date_only", to_date(col("timestamp"))))
display(load_df)

load_df.write.mode("overwrite").saveAsTable("energy_intelligence.silver.load_generation_15min")




In [0]:
%sql
CREATE OR REPLACE TABLE energy_intelligence.silver.wind_solar_forecast_joined
AS
SELECT 
    a.*, 
    CAST(period_start AS DATE) as start_date,
    CAST(period_end AS DATE) as end_date,
    b.*
FROM parquet.`/Volumes/energy_intelligence/bronze/landing_zone/data/entsoe/wind_solar_forecast/*.parquet` a
LEFT JOIN energy_intelligence.bronze.time_series_15_min b
ON a.position = b.counter;
